In [7]:
import pandas as pd

eia = pd.read_csv("data/raw/eia_energy_data.csv", parse_dates=["datetime_utc"])
print(eia.shape)
eia.head()

(17288, 7)


,datetime_utc,region,demand_mwh,hour,day_of_week,month,is_weekend
0,2026-03-15 22:00:00,BPAT,7608,22,6,3,1
1,2026-03-15 23:00:00,BPAT,7638,23,6,3,1
2,2026-03-16 00:00:00,BPAT,7784,0,0,3,0
3,2026-03-16 01:00:00,BPAT,7957,1,0,3,0
4,2026-03-16 02:00:00,BPAT,8097,2,0,3,0


In [1]:
%pip install openmeteo-requests requests-cache retry-requests


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 772.5/772.5 kB 10.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 618.6/618.6 kB 9.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 11.3 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12/12 [openmeteo-requests]iquests]uture]
Note: you may need to restart the kernel to use updated packages.


In [5]:
import pandas as pd

eia = pd.read_csv("../data/raw/eia_energy_data.csv", parse_dates=["datetime_utc"])
print(eia.shape)
eia.head()

(17288, 7)


,datetime_utc,region,demand_mwh,hour,day_of_week,month,is_weekend
0,2026-03-15 22:00:00,BPAT,7608,22,6,3,1
1,2026-03-15 23:00:00,BPAT,7638,23,6,3,1
2,2026-03-16 00:00:00,BPAT,7784,0,0,3,0
3,2026-03-16 01:00:00,BPAT,7957,1,0,3,0
4,2026-03-16 02:00:00,BPAT,8097,2,0,3,0


In [6]:
import openmeteo_requests
import requests_cache
from retry_requests import retry

# Setup connection to Open-Meteo with caching and retry
cache_session = requests_cache.CachedSession('.cache', expire_after=-1)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
openmeteo = openmeteo_requests.Client(session=retry_session)

# Region -> (city, latitude, longitude)
regions = {
    "BPAT": ("Seattle",      47.6062, -122.3321),
    "CISO": ("Sacramento",   38.5816, -121.4944),
    "ERCO": ("Dallas",       32.7767,  -96.7970),
    "ISNE": ("Boston",       42.3601,  -71.0589),
    "MISO": ("Chicago",      41.8781,  -87.6298),
    "NYIS": ("New York",     40.7128,  -74.0060),
    "PJM":  ("Philadelphia", 39.9526,  -75.1652),
    "SWPP": ("Oklahoma City",35.4676,  -97.5164),
}

START = "2026-03-15"
END   = "2026-06-13"

url = "https://archive-api.open-meteo.com/v1/archive"
all_weather = []

for region, (city, lat, lon) in regions.items():
    print(f"Fetching {city} for {region}...")

    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": START,
        "end_date": END,
        "hourly": "temperature_2m,apparent_temperature,relative_humidity_2m,precipitation",
    }

    responses = openmeteo.weather_api(url, params=params)
    response = responses[0]
    hourly = response.Hourly()

    hourly_data = {
        "datetime_utc": pd.date_range(
            start=pd.to_datetime(hourly.Time(), unit="s", utc=True),
            end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),
            freq=pd.Timedelta(seconds=hourly.Interval()),
            inclusive="left"
        ),
        "temperature":   hourly.Variables(0).ValuesAsNumpy(),
        "apparent_temp": hourly.Variables(1).ValuesAsNumpy(),
        "humidity":      hourly.Variables(2).ValuesAsNumpy(),
        "precipitation": hourly.Variables(3).ValuesAsNumpy(),
    }
    hourly_data["region"] = region

    df = pd.DataFrame(data=hourly_data)
    all_weather.append(df)

weather_df = pd.concat(all_weather, ignore_index=True)
weather_df["datetime_utc"] = weather_df["datetime_utc"].dt.tz_localize(None)

print("\nDone! Weather data shape:", weather_df.shape)
weather_df.head()

Fetching Seattle for BPAT...
Fetching Sacramento for CISO...
Fetching Dallas for ERCO...
Fetching Boston for ISNE...
Fetching Chicago for MISO...
Fetching New York for NYIS...
Fetching Philadelphia for PJM...
Fetching Oklahoma City for SWPP...

Done! Weather data shape: (17472, 6)


,datetime_utc,temperature,apparent_temp,humidity,precipitation,region
0,2026-03-15 00:00:00,6.85,4.705282,63.496994,0.0,BPAT
1,2026-03-15 01:00:00,7.15,4.588635,63.110184,0.0,BPAT
2,2026-03-15 02:00:00,6.40,3.482548,64.550583,0.0,BPAT
3,2026-03-15 03:00:00,5.75,2.607332,66.060349,0.0,BPAT
4,2026-03-15 04:00:00,5.35,2.095968,67.426262,0.0,BPAT


In [7]:
weather_df = weather_df.rename(columns={
    "temperature": "temperature_C",
    "apparent_temp": "apparent_temp_C",
    "humidity": "humidity_pct",
    "precipitation": "precipitation_mm"
})

weather_df.head()

,datetime_utc,temperature_C,apparent_temp_C,humidity_pct,precipitation_mm,region
0,2026-03-15 00:00:00,6.85,4.705282,63.496994,0.0,BPAT
1,2026-03-15 01:00:00,7.15,4.588635,63.110184,0.0,BPAT
2,2026-03-15 02:00:00,6.40,3.482548,64.550583,0.0,BPAT
3,2026-03-15 03:00:00,5.75,2.607332,66.060349,0.0,BPAT
4,2026-03-15 04:00:00,5.35,2.095968,67.426262,0.0,BPAT


In [8]:
merged = eia.merge(weather_df, on=["datetime_utc", "region"], how="left")

print("Merged shape:", merged.shape)
print("Missing weather values:", merged["temperature_C"].isna().sum())
merged.head()

Merged shape: (17288, 11)
Missing weather values: 0


,datetime_utc,region,demand_mwh,hour,day_of_week,month,is_weekend,temperature_C,apparent_temp_C,humidity_pct,precipitation_mm
0,2026-03-15 22:00:00,BPAT,7608,22,6,3,1,7.10,4.323064,63.326538,0.0
1,2026-03-15 23:00:00,BPAT,7638,23,6,3,1,7.10,4.528995,63.326538,0.1
2,2026-03-16 00:00:00,BPAT,7784,0,0,3,0,7.65,5.021635,66.480186,0.3
3,2026-03-16 01:00:00,BPAT,7957,1,0,3,0,7.35,4.412580,69.323738,0.4
4,2026-03-16 02:00:00,BPAT,8097,2,0,3,0,6.75,3.597219,69.448616,0.4


In [9]:
import os
os.makedirs("../data/processed", exist_ok=True)

merged.to_csv("../data/processed/eia_with_weather.csv", index=False)
print("Saved!")

Saved!


In [13]:
weather_df.to_csv("../data/raw/weather_data.csv", index=False)
print("Saved!")

Saved!


In [16]:
!cat ../environment.yml

name: ai4all-11a
channels:
  - conda-forge
  - defaults
dependencies:
  - python=3.11
  - numpy>=1.26
  - pandas>=2.2
  - requests>=2.32
  - python-dotenv>=1.0
  - matplotlib>=3.9
  - seaborn>=0.13
  - scikit-learn>=1.5
  - jupyterlab>=4.2
  - pip
  - pip:
      - openmeteo-requests
      - requests-cache
      - retry-requests

In [18]:
!cat ../requirements.txt

# Core data
numpy>=1.26
pandas>=2.2

# API
requests>=2.32
python-dotenv>=1.0

# Visualization
matplotlib>=3.9
seaborn>=0.13

# Supervised ML
scikit-learn>=1.5

# Notebooks
jupyterlab>=4.2

# Weather API
openmeteo-requests
requests-cache
retry-requests

In [19]:
!git log --oneline

f7eefeb (HEAD -> main) Added weather data from Open-Meteo and merged with EIA data
292d063 Added weather data from Open-Meteo and merged with EIA data
5908c54 (origin/main, origin/HEAD) instructions for setup
6bf64ef initial EDA and data pulled from EIA API
d4ee6a5 Update README.md
0e0ab3f Set up directory
0a13edf Initial commit


In [20]:
!git show --stat 292d063
!git show --stat f7eefeb

commit 292d063de1f74d2178d50188a630735aa5110d5f
Author: Leul Wolderufael <leulwolderufael@gmail.com>
Date:   Fri Jun 19 22:01:25 2026 -0700

    Added weather data from Open-Meteo and merged with EIA data

 notebooks/.cache.sqlite                            | Bin 0 -> 319488 bytes
 .../.ipynb_checkpoints/leul_eda-checkpoint.ipynb   |  33 +
 notebooks/leul_eda.ipynb                           | 845 +++++++++++++++++++++
 3 files changed, 878 insertions(+)
commit f7eefebb1d53b25847dcc3cf426b8e0dbc9ad30c (HEAD -> main)
Author: Leul Wolderufael <leulwolderufael@gmail.com>
Date:   Sat Jun 20 12:33:29 2026 -0700

    Added weather data from Open-Meteo and merged with EIA data

 notebooks/leul_eda.ipynb | 50 ++++++++++++++++++++++++++++++++++++++----------
 1 file changed, 40 insertions(+), 10 deletions(-)


In [21]:
!git reset --soft 5908c54

In [22]:
!git status

On branch main
Your branch is up to date with 'origin/main'.

Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	new file:   .cache.sqlite
	new file:   .ipynb_checkpoints/leul_eda-checkpoint.ipynb
	modified:   leul_eda.ipynb

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   ../environment.yml
	modified:   leul_eda.ipynb
	modified:   ../requirements.txt

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	../.DS_Store
	../.ipynb_checkpoints/
	../Miniconda3-latest-MacOSX-arm64.sh
	../data/processed/
	../data/raw/weather_data.csv



In [23]:
!git log --oneline -5


5908c54 (HEAD -> main, origin/main, origin/HEAD) instructions for setup
6bf64ef initial EDA and data pulled from EIA API
d4ee6a5 Update README.md
0e0ab3f Set up directory
0a13edf Initial commit


In [24]:
!git restore --staged .cache.sqlite
!git restore --staged .ipynb_checkpoints/leul_eda-checkpoint.ipynb

In [26]:
with open("../.gitignore", "a") as f:
    f.write("\nnotebooks/.cache.sqlite\nnotebooks/.ipynb_checkpoints/\n.ipynb_checkpoints/\n.DS_Store\n")

print("Updated!")

Updated!


In [27]:
!cat ../.gitignore

CLAUDE.md
.claude/
.env
notebooks/.cache.sqlite
notebooks/.ipynb_checkpoints/
.ipynb_checkpoints/
.DS_Store
